In [1]:
from datetime import date

import numpy as np
import pandas as pd
import yaml
from sqlalchemy import create_engine


# database connections 

In [2]:
with open('../config_fill.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_co = config['CO_SA']
    config_etl = config['ETL_PRO']

# Construct the database URL
url_co = (f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:"
          f"{config_co['port']}/{config_co['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")
# Create the SQLAlchemy Engine
co_sa = create_engine(url_co)
etl_conn = create_engine(url_etl)

# Extract

In [4]:
t_clientes_mensajeroaquitoy = pd.read_sql_table('clientes_mensajeroaquitoy', co_sa)
t_auth_user= pd.read_sql_table('auth_user', co_sa)
#dim_mensajero = pd.read_sql_table('clientes_mensajeroaquitoy', co_sa)


OperationalError: (psycopg2.OperationalError) connection to server at "localhost" (::1), port 5432 failed: FATAL:  database "fast_and_safe_OLTP" does not exist

(Background on this error at: https://sqlalche.me/e/20/e3q8)

# Transformations

In [ ]:

t_clientes_mensajeroaquitoy.drop(columns=["activo", "fecha_entrada", "fecha_salida", "salario", "token_Firebase", "url_foto"], inplace=True, errors="ignore")
t_auth_user.drop(columns=["password", "last_login", "is_superuser", "username","email","is_staff", "is_active", "date_joined"], inplace=True, errors="ignore")

In [ ]:
t_clientes_mensajeroaquitoy.info()
t_auth_user.info()

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   50 non-null     int64  
 1   user_id              50 non-null     int64  
 2   telefono             50 non-null     str    
 3   ciudad_operacion_id  45 non-null     float64
dtypes: float64(1), int64(2), str(1)
memory usage: 1.7 KB
<class 'pandas.DataFrame'>
RangeIndex: 329 entries, 0 to 328
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   id          329 non-null    int64
 1   first_name  329 non-null    str  
 2   last_name   329 non-null    str  
dtypes: int64(1), str(2)
memory usage: 7.8 KB


In [ ]:
t_clientes_mensajeroaquitoy.rename(columns={'id': 'id_mensajero','ciudad_operacion_id': 'id_ciudad_operacion','user_id': 'id_usuario'}, inplace=True)
dim_mensajero = pd.merge(t_clientes_mensajeroaquitoy, t_auth_user, left_on="id_usuario", right_on="id", how="left")
dim_mensajero.replace({np.nan: 'no aplica', ' ': 'no aplica','':'no_aplica'}, inplace=True)
dim_mensajero.rename(columns={'first_name': 'nombre','last_name': 'apellido'}, inplace=True)
dim_mensajero.drop(columns=["id","id_ciudad_operacion"], inplace=True, errors="ignore")

In [ ]:
dim_mensajero.info()

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   id_mensajero         50 non-null     int64 
 1   id_usuario           50 non-null     int64 
 2   telefono             50 non-null     str   
 3   id_ciudad_operacion  50 non-null     object
 4   id                   50 non-null     int64 
 5   first_name           50 non-null     str   
 6   last_name            50 non-null     str   
dtypes: int64(3), object(1), str(3)
memory usage: 2.9+ KB


# load

In [ ]:
dim_mensajero.head()

,id,user_id,activo,fecha_entrada,fecha_salida,salario,telefono,ciudad_operacion_id,token_Firebase,url_foto
0,1,2,True,no aplica,no aplica,no aplica,310-300000,13.0,no aplica,http:
1,42,410,True,no aplica,no aplica,no aplica,310-300000,1.0,no aplica,http:
2,48,447,True,2024-07-12 00:00:00,no aplica,1300000.0,310-300000,1.0,no aplica,http:
3,41,400,True,no aplica,no aplica,no aplica,310-300000,1.0,no aplica,http:
4,13,331,True,2021-11-08 00:00:00,no aplica,1160000.0,310-300000,4.0,no aplica,http:


In [ ]:
dim_mensajero.to_sql('dim_mensajero', etl_conn, if_exists='replace',index_label='key_dim_mensajero')

50